# SQL Query Validation & Analytics

## Bluestock Mutual Fund Analytics Capstone

### Objective
This notebook validates the SQLite database by executing analytical SQL queries on the cleaned and loaded datasets.

### Database
- SQLite Database: bluestock_mf.db
- Schema: Star Schema + Staging Tables
- Validation Date: Day 2

### Queries Covered
1. Top 5 Fund Houses by AUM
2. Average NAV by Fund
3. Average NAV Per Month
4. SIP Inflow YoY Growth
5. Transactions by State
6. Funds with Expense Ratio Below 1%
7. Top 10 Funds by 5-Year Return
8. Average Transaction Amount by Type
9. Portfolio Allocation by Sector
10. Highest Sharpe Ratio Funds

In [38]:
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine

BASE_DIR = Path.cwd().parent

DB_PATH = (
    BASE_DIR /
    "data" /
    "db" /
    "bluestock_mf.db"
)

engine = create_engine(
    f"sqlite:///{DB_PATH}"
)

print("Database Connected Successfully")

Database Connected Successfully


## Query 1: Top 5 Fund Houses by AUM

### Expected Insight
Identify the largest mutual fund companies based on total AUM.

In [39]:
query = """
SELECT
fund_house,
SUM(aum_crore) AS total_aum
FROM aum
GROUP BY fund_house
ORDER BY total_aum DESC
LIMIT 5;
"""

pd.read_sql(query, engine)

,fund_house,total_aum
0,SBI Mutual Fund,8491000
1,ICICI Prudential MF,6293000
2,HDFC Mutual Fund,5732000
3,Nippon India MF,3909000
4,Kotak Mahindra MF,3502000


## Query 2: Average NAV by Fund

### Expected Insight
Compare average scheme valuations.

In [40]:
query = """
SELECT
amfi_code,
AVG(nav) AS avg_nav
FROM nav_history
GROUP BY amfi_code;
"""

pd.read_sql(query, engine)

,amfi_code,avg_nav
0,100016,567.776731
1,100025,29.680419
2,100033,218.278644
3,101206,486.998783
4,101207,59.784547
5,101208,367.856690
6,102885,138.582522
7,102886,133.407849
8,102887,269.930680
9,118632,80.498568


## Query 3: Average NAV Per Month

### Expected Insight
Understand NAV trends across months.

In [41]:
query = """
SELECT
strftime('%Y-%m', date) AS month,
AVG(nav) AS avg_nav
FROM nav_history
GROUP BY month
ORDER BY month;
"""

pd.read_sql(query, engine)

,month,avg_nav
0,2022-01,221.304573
1,2022-02,221.354099
2,2022-03,218.794191
3,2022-04,218.861727
4,2022-05,221.105809
5,2022-06,220.956152
6,2022-07,221.269027
7,2022-08,221.812133
8,2022-09,222.054156
9,2022-10,219.099683


## Query 4: SIP Inflow YoY Growth

### Expected Insight
Measure growth in systematic investments.

In [42]:
query = """
SELECT
month,
sip_inflow_crore,
yoy_growth_pct
FROM sip
ORDER BY month;
"""

pd.read_sql(query, engine)

,month,sip_inflow_crore,yoy_growth_pct
0,2022-01,11517,NaN
1,2022-02,11438,NaN
2,2022-03,12328,NaN
3,2022-04,11863,NaN
4,2022-05,12286,NaN
5,2022-06,12276,NaN
6,2022-07,12140,NaN
7,2022-08,12694,NaN
8,2022-09,12976,NaN
9,2022-10,13040,NaN


## Query 5: Transactions by State

### Expected Insight
Identify regions with high investor participation.

In [43]:
query = """
SELECT
state,
COUNT(*) AS transactions
FROM transactions
GROUP BY state
ORDER BY transactions DESC;
"""

pd.read_sql(query, engine)

,state,transactions
0,Punjab,2965
1,Madhya Pradesh,2931
2,Tamil Nadu,2806
3,Gujarat,2780
4,West Bengal,2748
5,Haryana,2736
6,Telangana,2718
7,Uttar Pradesh,2695
8,Delhi,2677
9,Karnataka,2621


## Query 6: Funds with Expense Ratio Below 1%

### Expected Insight
Identify cost-efficient investment options.

In [44]:
query = """
SELECT
scheme_name,
expense_ratio_pct
FROM performance
WHERE expense_ratio_pct < 1
ORDER BY expense_ratio_pct;
"""

pd.read_sql(query, engine)

,scheme_name,expense_ratio_pct
0,Nippon India Gilt Securities Fund - Regular - ...,0.55
1,HDFC Short Term Debt Fund - Regular - Growth,0.56
2,Kotak Liquid Fund - Regular - Growth,0.60
3,SBI Bluechip Fund - Direct Plan - Growth,0.66
4,SBI Small Cap Fund - Direct Plan - Growth,0.72
5,Nippon India Large Cap Fund - Direct - Growth,0.72
6,ICICI Pru Liquid Fund - Regular - Growth,0.74
7,Axis Bluechip Fund - Direct - Growth,0.75
8,SBI Magnum Gilt Fund - Regular Plan - Growth,0.77
9,HDFC Mid-Cap Opportunities Fund - Direct - Growth,0.78


## Query 7: Top 10 Funds by 5-Year Return

### Expected Insight
Identify top-performing mutual funds.

In [45]:
query = """
SELECT
scheme_name,
return_5yr_pct
FROM performance
ORDER BY return_5yr_pct DESC
LIMIT 10;
"""

pd.read_sql(query, engine)

,scheme_name,return_5yr_pct
0,ABSL Small Cap Fund - Regular - Growth,23.80
1,Axis Small Cap Fund - Regular - Growth,22.62
2,Nippon India Small Cap Fund - Regular - Growth,21.88
3,SBI Small Cap Fund - Direct Plan - Growth,21.82
4,SBI Small Cap Fund - Regular Plan - Growth,20.67
5,DSP Small Cap Fund - Regular - Growth,20.61
6,DSP Midcap Fund - Regular - Growth,19.00
7,Axis Midcap Fund - Regular - Growth,18.94
8,Kotak Emerging Equity Fund - Regular - Growth,17.75
9,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,17.69


## Query 8: Average Transaction Amount by Type

### Expected Insight
Compare SIP, Lumpsum and Redemption behaviour.

In [46]:
query = """
SELECT
transaction_type,
AVG(amount_inr) AS avg_amount
FROM transactions
GROUP BY transaction_type;
"""

pd.read_sql(query, engine)

,transaction_type,avg_amount
0,Lumpsum,254456.015812
1,Redemption,250558.786189
2,SIP,11018.132025


## Query 9: Portfolio Allocation by Sector

### Expected Insight
Understand sector concentration across schemes.

In [47]:
query = """
SELECT
sector,
SUM(weight_pct) AS total_weight
FROM holdings
GROUP BY sector
ORDER BY total_weight DESC;
"""

pd.read_sql(query, engine)

,sector,total_weight
0,Banking,652.26
1,IT,455.47
2,Pharma,407.45
3,Automobile,323.65
4,Utilities,265.54
5,FMCG,229.11
6,Infrastructure,192.16
7,Diversified,169.23
8,Telecom,145.62
9,Consumer Goods,127.61


## Query 10: Highest Sharpe Ratio Funds

### Expected Insight
Identify schemes with superior risk-return characteristics.

In [48]:
query = """
SELECT
scheme_name,
sharpe_ratio
FROM performance
ORDER BY sharpe_ratio DESC
LIMIT 10;
"""

pd.read_sql(query, engine)

,scheme_name,sharpe_ratio
0,ICICI Pru Liquid Fund - Regular - Growth,7.68
1,Kotak Liquid Fund - Regular - Growth,6.18
2,ABSL Liquid Fund - Regular - Growth,5.14
3,HDFC Short Term Debt Fund - Regular - Growth,1.84
4,SBI Magnum Gilt Fund - Regular Plan - Growth,1.52
5,Nippon India Gilt Securities Fund - Regular - ...,1.33
6,HDFC Top 100 Fund - Regular Plan - Growth,1.06
7,Mirae Asset Large Cap Fund - Regular - Growth,1.06
8,ICICI Pru Bluechip Fund - Direct - Growth,1.03
9,Nippon India Large Cap Fund - Regular - Growth,1.00


# SQL Validation Summary

## Key Outcomes

- Successfully connected to SQLite database.
- Executed 10 analytical SQL queries.
- Verified database tables and relationships.
- Generated insights on:
  - Fund AUM
  - NAV trends
  - SIP growth
  - Investor transactions
  - Expense ratios
  - Fund performance
  - Sector allocations
  - Risk-adjusted returns

## Conclusion

The SQLite database was successfully validated and supports analytical reporting requirements for the Mutual Fund Analytics Capstone.